In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
from googleapiclient.discovery import build
from google.oauth2 import service_account

In [2]:
import datetime
import calendar

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el primer día del mes actual y restar un día para obtener el mes anterior
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - datetime.timedelta(days=1)

# Extraer mes y año del mes anterior
mes_anterior = ultimo_dia_mes_anterior.month
anio_anterior = ultimo_dia_mes_anterior.year

# Asignar variables
MES_A_LIQUIDAR = f"{mes_anterior:02d}/{anio_anterior}"
PERIODO = f"{anio_anterior}{mes_anterior:02d}"

# Imprimir resultados
print("MES_A_LIQUIDAR =", MES_A_LIQUIDAR)
print("Periodo =", PERIODO)

MES_A_LIQUIDAR = 10/2026
Periodo = 202610


In [4]:
# Autenticación con la clave de servicio
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
KEY = 'key2.json'  # LLAVES
SPREADSHEET_ID = "1cfH0XqAuofeJDB6QDb4fxhu3ixSeN8g_pOVyUlJ1g4k"  # ID DE LA HOJA DE CALCULO

# Crear credenciales
creds = service_account.Credentials.from_service_account_file(KEY, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()

# Llamar a la API de Google Sheets
RANGE_NAME = f"'{PERIODO}'!A1:F1000"  # HOJA Y RANGO A TRABAJAR

result = sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=RANGE_NAME).execute()
values = result.get('values', [])

# Mostrar resultados
if not values:
    print("No se encontraron datos.")
else:
    for row in values:
        print("Datos obtenidos")


afectaciones = pd.DataFrame(values[1:], columns=values[0])


Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos obtenidos
Datos ob

In [5]:
afectaciones

,CC GESTOR,CORREO AGENTE,CALIDAD,ADHERENCIA,CONVERSIÓN,PERIODO
0,1002698832,adriana_medinac@cun.edu.co,Cumple,No cumple,No cumple,202610
1,1022422812,alejandro_bueno@cun.edu.co,Cumple,Cumple,No cumple,202610
2,52067229,alexandra_gonzalez@cun.edu.co,Cumple,No cumple,Cumple,202610
3,1000004014,alisson_ramirezg@cun.edu.co,Cumple,No cumple,No cumple,202610
4,14327321,anderson_crispinp@cun.edu.co,Cumple,Cumple,Cumple,202610
...,...,...,...,...,...,...
106,1007298501,saray_jimenez@cun.edu.co,Cumple,Cumple,No cumple,202610
107,1012337219,sol_reyes@cun.edu.co,Cumple,Cumple,No cumple,202610
108,1024589527,ingrit_camargo@cun.edu.co,Cumple,Cumple,No cumple,202610
109,1032477608,angie_bolanos@cun.edu.co,Cumple,Cumple,No cumple,202610


In [6]:
afectaciones['PERIODO'].unique()

array(['202610'], dtype=object)

In [7]:
afectaciones = afectaciones.fillna('')

In [8]:
afectaciones = afectaciones.astype(str)
afectaciones

,CC GESTOR,CORREO AGENTE,CALIDAD,ADHERENCIA,CONVERSIÓN,PERIODO
0,1002698832,adriana_medinac@cun.edu.co,Cumple,No cumple,No cumple,202610
1,1022422812,alejandro_bueno@cun.edu.co,Cumple,Cumple,No cumple,202610
2,52067229,alexandra_gonzalez@cun.edu.co,Cumple,No cumple,Cumple,202610
3,1000004014,alisson_ramirezg@cun.edu.co,Cumple,No cumple,No cumple,202610
4,14327321,anderson_crispinp@cun.edu.co,Cumple,Cumple,Cumple,202610
...,...,...,...,...,...,...
106,1007298501,saray_jimenez@cun.edu.co,Cumple,Cumple,No cumple,202610
107,1012337219,sol_reyes@cun.edu.co,Cumple,Cumple,No cumple,202610
108,1024589527,ingrit_camargo@cun.edu.co,Cumple,Cumple,No cumple,202610
109,1032477608,angie_bolanos@cun.edu.co,Cumple,Cumple,No cumple,202610


In [9]:
afectaciones.dtypes

CC GESTOR        object
CORREO AGENTE    object
CALIDAD          object
ADHERENCIA       object
CONVERSIÓN       object
PERIODO          object
dtype: object

In [10]:
afectaciones = afectaciones.rename(columns={'CC GESTOR ' : 'CC GESTOR'})

In [11]:
afectaciones.rename(columns={
    'CC GESTOR': 'Documento_del_Asesor',
    'CORREO AGENTE': 'Correo_Asesor',
    'CONVERSIÓN':'CONVERSION'
}, inplace=True)

In [12]:
afectaciones_Columnas = ['FORMACION', 'VENTAS', 'PRODUCTIVIDAD','CALIDAD','ADHERENCIA','CONVERSION']

for col in afectaciones_Columnas:
    if col not in afectaciones.columns:
        afectaciones[col] = None

In [13]:
# Parámetros de conexión
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)

# Crear el motor de conexión
engine = create_engine(url)

# Nombre de la tabla y esquema
table_name = 'Afectaciones_Contact'
schema_name = 'Comisiones_Contact'

try:
    with engine.connect() as conn:
        # Realizar inserción por lotes usando `to_sql` con `if_exists='append'`
        afectaciones.to_sql(
            name=table_name,
            schema=schema_name,
            con=engine,
            if_exists='append',
            index=False,
            chunksize = 10000
        )
    print(f"Datos insertados correctamente en la tabla {schema_name}.{table_name}")
except Exception as e:
    print("Error al insertar los datos:", e)
finally:
    engine.dispose()

Datos insertados correctamente en la tabla Comisiones_Contact.Afectaciones_Contact
